In [59]:
import pandas as pd
import numpy as np
import sys
import glob
from scipy.ndimage import median_filter
from scipy import signal
import math
import matplotlib.pyplot as plt
import mwdust
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.interpolate import CubicSpline

In [3]:
sys.path.append("/Users/pnr5sh/Documents/phd/mmmp/")
import sidchaini.sidhelpers as sidhelpers

In [4]:
#reading in meta data from my dir
dataset = pd.read_csv('multipeak_dataset_metadata.csv', header='infer')
dataset['ztf_name'] = ['ZTF']*len(dataset)
dataset.columns

Index(['wise_objid', 'IAU name', 'Internal name/s', 'Obj. RA', 'Obj. DEC',
       'Obj. Type', 'Redshift', 'Spec. ID', 'Obs-date', 'JD', 'Phase (days)',
       'From', 'Telescope', 'Instrument', 'Observer/s', 'Reducer/s',
       'Source group', 'Public', 'Associated groups', 'End prop. period',
       'Ascii file', 'Fits file', 'Spec. type', 'Spec. quality',
       'Extinction-Corrected', 'WL Medium', 'WL Units',
       'Flux Unit Coefficient', 'Spec. units', 'Flux Calibrated By',
       'Exp-time', 'Aperture (slit)', 'HA', 'Airmass', 'Dichroic', 'Grism',
       'Grating', 'Blaze', 'Lambda-min', 'Lambda-max', 'Del-Lambda', 'Contrib',
       'Publish', 'Remarks', 'Created by', 'Creation date', 'mjd', 'peak_mjd',
       'peak_mag', 'peak_filt', 'double-peaked', 'ztf_name'],
      dtype='object')

In [5]:
#manually renaming objects w/o pre-loaded ZTF names
dataset.loc[dataset['IAU name']=='SN 2024zsw', 'Internal name/s'] = 'ZTF24abpdzvm'
dataset.loc[dataset['IAU name']=='SN 2020urc', 'Internal name/s'] = 'ZTF20acgiglu'
# dataset.loc[dataset['IAU name']=='SN 2018cew', 'Internal name/s'] = 'ZTF...' #doesnt have name?
dataset.loc[dataset['IAU name']=='SN 2024abmk', 'Internal name/s'] = 'ZTF24absznoi'
dataset.loc[dataset['IAU name']=='SN 2024abtu', 'Internal name/s'] = 'ZTF24abtnkbi'
dataset.loc[dataset['IAU name']=='SN 2024zzy', 'Internal name/s'] = 'ZTF24abqqven'

In [6]:
#populating the ztf_name feature of the dataset by extracting from internal name/s feature

intnamelist = dataset['Internal name/s'].to_list()
iaunames = dataset['IAU name'].to_list()

name_list = [(str(intnamelist[i]), str(iaunames[i])) for i in range(len(intnamelist))]

for pair in name_list:
    intname = pair[0]
    iauname = pair[1]
    varX = "ZTF"
    index = intname.find(varX)
    ztf_name = intname[index:index+12]
    if ztf_name[0] != 'Z':
        print(f'warning: could not parse out ZTF name from internal names list for {iauname}')
    else:
        dataset.loc[dataset['IAU name']==iauname, 'ztf_name'] = ztf_name

In [ ]:
#creating our version of the ZTF BTS metadata info file
# ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,peakabs,duration,rise,fade,type,redshift,b,A_V
# already have: ZTFID,IAUID,RA,Dec,peakt,peakfilt,peakmag,type,redshift, double-peaked
# want to get: A_V, [could calc peakabs if want, fits weren't necessarily constrained]
# ignoring: duration, rise, fade, b, peakabs

#?TODO: calc abs mag (hubble flow?)

bs_ZTFBTS = pd.DataFrame(columns=['ZTFID','IAUID','RA','Dec','peakt','peakfilt',
                                  'peakmag','type','redshift','double-peaked'])

znames, iaunames, ras, decs, pmjds, pfilts, pmags, types, zs, dps = [],[],[],[],[],[],[],[],[],[]
for i,obj in enumerate(dataset['ztf_name'].unique()):
    if obj=='ZTF':
        print(f'No ZTF name, skipping {dataset.loc[dataset['ztf_name']=='ZTF', 'IAU name'].iloc[0]} for now...')
    else:
        data = dataset.loc[dataset['ztf_name']==obj]
        
        #some objs have repeat entries, just taking the first one
        znames.append(data['ztf_name'].iloc[0])
        iaunames.append(data['IAU name'].iloc[0][0:2]+data['IAU name'].iloc[0][3:])
        ras.append(data['Obj. RA'].iloc[0])
        decs.append(data['Obj. DEC'].iloc[0])
        pmjds.append(data['peak_mjd'].iloc[0])
        pfilts.append(data['peak_filt'].iloc[0][-1])
        pmags.append(data['peak_mag'].iloc[0])
        types.append(data['Obj. Type'].iloc[0])
        zs.append(data['Redshift'].iloc[0])
        dps.append(data['double-peaked'].iloc[0])
        

bs_ZTFBTS['ZTFID'] = znames
bs_ZTFBTS['IAUID'] = iaunames
bs_ZTFBTS['RA'] = ras
bs_ZTFBTS['Dec'] = decs
bs_ZTFBTS['peakt'] = pmjds
bs_ZTFBTS['peakfilt'] = pfilts
bs_ZTFBTS['peakmag'] = pmags
bs_ZTFBTS['type'] = types
bs_ZTFBTS['redshift'] = zs
bs_ZTFBTS['double-peaked'] = dps

No ZTF name, skipping SN 2018cew for now...


In [ ]:
#calculating extinction
r_v = 3.1

avs = []
for i in range(len(bs_ZTFBTS)):
    sfd = mwdust.SFD()
    coord = SkyCoord(bs_ZTFBTS.RA.iloc[i] * u.deg, bs_ZTFBTS.Dec.iloc[i] * u.deg).galactic
    av = r_v*sfd(coord.l.value, coord.b.value, 5000)[0]
    avs.append(av)

bs_ZTFBTS['A_V'] = avs

In [39]:
bs_ZTFBTS.to_csv('./maven_data/bs_ZTFBTS_TransientTable.csv', index=False, index_label=False)

In [ ]:
#maven needs light curves files w/ 4 columns: time,mag,magerr,band
#   where time is in mjd and filter is either g, R

# dropping bad obs
# centering LCs around date of first spectra (-50, +150 days)
# creating mag and mag_err cols
# dropping points w/ e_mag>2 mags
# creating mjd col

def create_ztf_lc_dfs(datadir, sn_type, check_names=False, save_df=True):
        files = sorted(glob.glob(datadir+'*_fp_lc.txt'))
        ztf_info_df = pd.read_csv(f'ztf_info_files/ztf_fp_info_for_{sn_type}.csv')

        if (sn_type == 'SN IIb') or (sn_type =='SN IIn') or (sn_type == 'SN Ibc'):
                startindx = 18
        elif (sn_type == 'SN Ic') or (sn_type =='SN Ib'):
                startindx = 17
        elif (sn_type == 'SLSN-I') or (sn_type == 'SN IbnIcn') or (sn_type == 'SN Ic-pec'):
                startindx = 20
        elif (sn_type == 'SLSN-II') or (sn_type =='SN Ibc Ca-rich'):
                startindx = 21
        elif sn_type == 'FBOT':
                startindx = 19
        else:
                print(f'SN type {sn_type} unsupported or not in format SN XX / SLSN-XX')
                return

        sn_names, lc_dfs = [],[]
        for file in files:
                sn_name = file[startindx:-10] #NOTE: FIRST INDEX CHANGES W/ DATADIR NAME LENGTH
                if check_names:
                        print(sn_name)
                        continue
                
                sn_names.append(sn_name)

                cols = ['index', 'field', 'ccdid', 'qid', 'filter', 'pid', 'infobitssci', 'sciinpseeing', 'scibckgnd', 'scisigpix',
                        'zpmaginpsci', 'zpmaginpsciunc', 'zpmaginpscirms', 'clrcoeff', 'clrcoeffunc', 'ncalmatches', 'exptime',
                        'adpctdif1', 'adpctdif2', 'diffmaglim', 'zpdiff', 'programid', 'jd', 'rfid', 'forcediffimflux', 'forcediffimfluxunc',
                        'forcediffimsnr', 'forcediffimchisq', 'forcediffimfluxap', 'forcediffimfluxuncap', 'forcediffimsnrap', 'aperturecorr',
                        'dnearestrefsrc', 'nearestrefmag', 'nearestrefmagunc', 'nearestrefchi', 'nearestrefsharp', 'refjdstart', 'refjdend', 'procstatus']
                df = pd.read_csv(file, names=cols, header=None, sep=" ", skiprows=54)
                df = df.set_index(df['index'])  # manually setting indeces
                df = df.drop(columns=['index']) # drop duplicated index 
                df = df[(df['infobitssci'] < 33554432) & (df['scisigpix'] <= 25) & (df['sciinpseeing'] <= 4) & (df['forcediffimflux']!=-99999.0)].reset_index(drop=True) #clean according to docs

                # cut df down to -50 days to +365 days centered on date of first spectra
                # time window included in ztf_fp_info*csv files for each obj type
                obj = ztf_info_df.loc[ztf_info_df['obj_name'].str[3:]==sn_name]
                start_jd = obj['jd_start'].iloc[0]
                end_jd = obj['jd_end'].iloc[0]
                df_cut = df.loc[(df['jd']<end_jd)&(df['jd']>start_jd)] #only selecting points that fall within specified time window
                df_cut = df_cut.reset_index(drop=True)
                df_cut = df_cut.infer_objects() #infering dtype of columns

                mag = df_cut['zpdiff'] - 2.5 * np.log10(df_cut['forcediffimflux'])
                sigma_mag = 1.0857 * df_cut['forcediffimfluxunc']/df_cut['forcediffimflux']
                df_cut['mag'] = mag
                df_cut['e_mag'] = sigma_mag
                df_cut = df_cut.loc[(df_cut['forcediffimflux']>0) & (df_cut['e_mag']<2)].reset_index(drop=True) #only selecting points w/ non-nan mags and errorbars less than 2 mags
                df_cut['mjd'] = df_cut['jd']-2400000.5

                # saving LC in maven-friendly format w/ ZTF name as file name
                # if obj has no ZTF internal name, saved w/ IAU name and will need to look up personally 
                #       and update the metadata
                if save_df:
                        if 'SN '+sn_name in dataset['IAU name'].to_list(): #only converting objs in our "good" sample
                                smol_df = df_cut[['mjd', 'mag', 'e_mag', 'filter']]
                                smol_df = smol_df.rename(columns={"mjd": "time", "e_mag": "magerr", "filter":"band"})
                                smol_df.loc[smol_df['band']=='ZTF_g', 'band'] = 'g'
                                smol_df.loc[smol_df['band']=='ZTF_r', 'band'] = 'R'

                                ztf_name = dataset.loc[dataset['IAU name']=='SN '+sn_name, 'ztf_name'].iloc[0]
                                if len(ztf_name)<4:
                                        print(f'WARNING: SN {sn_name} has no ZTF name; need for maven')
                                        smol_df.to_csv(f'maven_data/lightcurves/SN{sn_name}.csv',index=False)
                                else:
                                        smol_df.to_csv(f'maven_data/lightcurves/{ztf_name}.csv',index=False)                                                             

                lc_dfs.append(df_cut)

        return sn_names, lc_dfs

In [15]:
# sn_names_ib, lc_dfs_ib = create_ztf_lc_dfs('./ztf_fp_data/ib/','SN Ib')
# sn_names_ic, lc_dfs_ic = create_ztf_lc_dfs('./ztf_fp_data/ic/','SN Ic')
sn_names_iib, lc_dfs_iib = create_ztf_lc_dfs('./ztf_fp_data/iib/','SN IIb', check_names=False, save_df=True)
# sn_names_carich, lc_dfs_carich = create_ztf_lc_dfs('./ztf_fp_data/carich/','SN Ibc Ca-rich', check_names=False)
# sn_names_ibc, lc_dfs_ibc = create_ztf_lc_dfs('./ztf_fp_data/ibc/','SN Ibc', check_names=False)
# sn_names_ibncn, lc_dfs_ibncn = create_ztf_lc_dfs('./ztf_fp_data/ibncn/','SN IbnIcn', check_names=False)
# sn_names_icpec, lc_dfs_icpec = create_ztf_lc_dfs('./ztf_fp_data/icpec/','SN Ic-pec', check_names=False)

/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/pnr5sh/miniconda3/envs/astro2/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [129]:
("2023aew" in 'adrian/wiserep_spectra/iib_spectra/spectra/2023aew_2023-05-11_09-01-59_P60_SEDM_None.txt')

True

In [169]:
#creating empty dataframe 
header = ['wavelength', 'flux', 'fluxerr']
#spectra files are either names for ZTF name or IAU name
spectrafile = pd.read_csv('./wiserep_spectra/iib_spectra/spectra/ZTF18abojpnr_2458351.897_P60_SEDM_ZTF.ascii', 
                          delimiter=' ', skiprows=168, header=0, names=header)

no_spec = len(dataset.loc[dataset['ztf_name']=='ZTF18abojpnr', 'Del-Lambda']) #Del-Lambda is wavelength spacing


# only want to preproc spectra for objs in our good sample, which are denoted by those in dataset df
raw_spec_file_names = glob.glob('./wiserep_spectra/iib_spectra/spectra/*')
sntype = (dataset['Obj. Type']=='SN IIb')

iau_str = dataset.loc[sntype, 'IAU name'].unique()
ztf_str = dataset.loc[sntype, 'ztf_name'].unique()

#generating list of 3 potential strings that spectra file could use for each obj
# wiserep names are either XXXXabc, SNXXXXabc, or ZTFabcdefg
potential_names = []
for i,iauname in enumerate(iau_str):
    sniau = iauname[0:2]+iauname[3:]
    iau = iauname[3:]
    ztf = ztf_str[i]
    potential_names.append([sniau, iau, ztf])

#check each spectra file to see if matches any of the 3 strs of our target objs
#appending each spectrum to list for each obj name, then appending all objs to 1 list
spectra_files_for_type = []
for i,name in enumerate(potential_names):
    sfiles = []
    for j,file in enumerate(raw_spec_file_names):
        if any(n in file for n in name):
            sfiles.append(file)
    spectra_files_for_type.append(sfiles)

#each file has different number of header lines, creating function that ignores commented lines
headerstart = "#"
def skip_to(fle, headerstart, **kwargs):
    with open(fle) as f:
        pos = 0
        curline = f.readline()
        while curline.startswith(headerstart):
            pos = f.tell()
            curline = f.readline()
        f.seek(pos)
        return pd.read_csv(f, **kwargs)

In [ ]:
#values taken from DASH paper
w0 = 3500
w1 = 10000
nw = 1024
w_den = (w1 - w0)/nw 
smooth = 6
dwlog = np.log(w1/w0)/nw #0.00102
a = nw / np.log(w1/w0) #975.40
b = -nw*np.log(w0)/np.log(w1/w0) #-7959.796

def preproc_spectra4maven(spectrafiles, plot=False, save=False, savedir='./maven_data/spectra/'):
    """
    preprocess spectra according to MAVEN paper
    1) apply low pass median filter to smooth spectrum
    2) create log-spaced wavelength bins, bin the flux
    3) fit 13-point cubic spline, divide it from spectrum to correct for continuum
    
    Returns list of preproc'd spectra
    """
    no_spec = len(spectrafiles)

    preproc_spectra = []
    for i in range(no_spec):
        #read in spectrum as df
        spectrafile = skip_to(spectrafiles[i], headerstart, header=0, names=header, delimiter=' ')

        #move the spectra to have mean = 0
        mean = np.mean(spectrafile['flux'])
        flux_cen0 = spectrafile['flux']-mean

        # set values in all columns, with wavelength outside of 3500-10000, to 0
        spectrafile.loc[(spectrafile['wavelength']<3500)|(spectrafile['wavelength']>10000), ['']] = 0

        #apply low-pass median filter to smooth spectrum
        del_lam = dataset.loc[dataset['ztf_name']=='ZTF18abojpnr', 'Del-Lambda'].iloc[i]
        window_size = math.ceil( (w_den / del_lam) * smooth ) #rounds value to next highest integer
        smooth_spec = spectrafile['flux'].rolling(window_size, center=True).median()
        spectrafile['smooth_flux'] = smooth_spec
        
        #log-spacing wavelength
        log_wave = np.logspace(np.log(3500), np.log(10000), num=1024, endpoint=True, base=np.e)
        log_wave_mid, binned_fluxes = [],[]
        for j, rightbin in enumerate(log_wave[1:]):
            leftbin = log_wave[j]
            binned_flux = spectrafile.loc[(leftbin<spectrafile['wavelength'])&
                                        (spectrafile['wavelength']<rightbin),
                                        'smooth_flux'].sum()
            binned_fluxes.append(binned_flux)
            log_wave_mid.append((leftbin+rightbin)/2)

        # for j in range(nw+1): 
        #     wlogn = w0*np.log(j*dwlog)
        #     binned_wave = a*np.log(wlogn)+b
            # print(j, wlogn, np.log(wlogn), a*np.log(wlogn), 3500+binned_wave)
            # binned_waves.append(binned_wave)

        # fitting and dividing out 13-point cubic spline fit to remove continuum
        non0_idx, non0_wave, non0_flux = [],[],[]
        for i,f in enumerate(binned_fluxes):
            if f != 0:
                non0_idx.append(i)
                non0_wave.append(log_wave_mid[i])
                non0_flux.append(f)
        
        idx = np.round(np.linspace(0, len(non0_flux) - 1, 13)).astype(int)
        cs_wave = [non0_wave[i] for i in idx]
        cs_flux = [non0_flux[i] for i in idx]

        cs = CubicSpline(cs_wave,cs_flux)

        cont_divid_flux = np.array(non0_flux) / cs(non0_wave)

        #normalizing to be b/w 0-1
        norm_flux = (cont_divid_flux-min(cont_divid_flux)) / (max(cont_divid_flux)-min(cont_divid_flux))
        
        if plot:
            plt.figure()
            plt.plot(non0_wave, norm_flux)
            plt.title('preproc spectrum')
            plt.show()
        
        #updating whole spectrum w/ norm'd values
        for i,idx in enumerate(non0_idx):
            binned_fluxes[idx] = norm_flux[i]
        
        #exporting smooth, binned, continuum-divided, normalized spectrum
        preproc_spectra.append([log_wave_mid, binned_fluxes])

        # if save:
        #     #save just as ZTFname.csv with 2 or 3 columns: ["freq", "spec", "specerr"]
        #     ...
    
    return preproc_spectra

In [171]:
for i in range(len(spectra_files_for_type)):
    sfiles = spectra_files_for_type[i]
    preproc_spectra4maven(sfiles, save=False, savedir='./maven_data/spectra/')

IndexError: single positional indexer is out-of-bounds